# Tinh chỉnh Trọng số Mô hình Thống Kê (ARIMA Family)

Notebook này tìm ra cấu hình tốt nhất cho các mô hình thống kê: **ARIMA, SARIMA, ARIMAX, SARIMAX**.
Các bước thực hiện:
1. **Lựa chọn Biến Ngoại sinh (Exogenous Feature Selection):** Sử dụng Spearman Correlation và Granger Causality.
2. **Kiểm tra Tính dừng (Stationarity):** Áp dụng Augmented Dickey-Fuller (ADF) Test.
3. **Auto-ARIMA (Hyperparameter Tuning):** Dùng thuật toán Grid Search (hoặc Stepwise) dựa trên tiêu chuẩn AIC.
4. **Kiểm định phần dư (Residual Diagnostics):** Sử dụng Ljung-Box Test để xác thực mô hình.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import pickle

from scipy.stats import spearmanr
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
import pmdarima as pm
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path("../outputs/predictions")
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODELING_DIR = Path("../data/processed/modeling_fs")
TARGET_LOG = "target_pm25_h24_log"


In [2]:
def load_xy(path: Path):
    df = pd.read_csv(path, parse_dates=["datetime_local"])
    feat = [c for c in df.columns if c not in ("datetime_local", TARGET_LOG)]
    return df[feat], df[TARGET_LOG], df["datetime_local"]

train_X, train_y, train_dt = load_xy(MODELING_DIR / "train_dl.csv")
val_X, val_y, val_dt = load_xy(MODELING_DIR / "val_dl.csv")
test_X, test_y, test_dt = load_xy(MODELING_DIR / "test_dl.csv")

print(f"Train shapes: X={train_X.shape}, y={train_y.shape}")
print(f"Val shapes: X={val_X.shape}, y={val_y.shape}")
print(f"Test shapes: X={test_X.shape}, y={test_y.shape}")


Train shapes: X=(6383, 26), y=(6383,)
Val shapes: X=(2128, 26), y=(2128,)
Test shapes: X=(2128, 26), y=(2128,)


## 1. Exogenous Feature Selection (Dành cho ARIMAX & SARIMAX)

### 1.1 Spearman Correlation
Tìm các biến (ngoại sinh) có tương quan đơn điệu tốt nhất với biến mục tiêu.

In [3]:
spearman_corr = {}
for col in train_X.columns:
    if "pm25" in col: # Loại bỏ các cột auto-correlation của PM2.5 vì mô hình ARIMA đã tự xử lý (AR)
        continue
    corr, pval = spearmanr(train_X[col], train_y)
    spearman_corr[col] = abs(corr)

corr_df = pd.DataFrame.from_dict(spearman_corr, orient='index', columns=['Spearman_Abs']).sort_values(by='Spearman_Abs', ascending=False)
display(corr_df.head(10))

# Lấy Top 5 biến weather tốt nhất để test vấp Granger
top_candidates = corr_df.head(5).index.tolist()
print("Top candidates:", top_candidates)


,Spearman_Abs
month_cos,0.468549
surface_pressure,0.393530
is_dry_season,0.360527
wind_u,0.312185
month,0.295046
temperature_2m,0.214018
relative_humidity_2m,0.109109
hour_cos,0.102689
hours_since_last_rain,0.053740
day_of_week,0.031981


Top candidates: ['month_cos', 'surface_pressure', 'is_dry_season', 'wind_u', 'month']


### 1.2 Granger Causality Test
Kiểm tra xem liệu các biến thời tiết có thực sự "gây ra" hay mang theo tín hiệu tiên lượng giá trị PM2.5 trong tương lai hay không. Ta sẽ kiểm chứng độ trễ từ 1 đến 24 giờ.

In [4]:
selected_exog = []
print("--- Kết quả Granger Causality (Tìm p-value < 0.05) ---")

for col in top_candidates:
    # Granger Causality yêu cầu DataFrame có 2 cột: [Cột mục tiêu (y), Cột nguyên nhân (X)]
    data = pd.concat([train_y, train_X[col]], axis=1)
    
    # maxlag=24 do ta quan tâm đến chu kỳ 1 ngày
    print(f"\nThử nghiệm: {col} -> PM2.5")
    try:
        gc_res = grangercausalitytests(data, maxlag=[1, 6, 12, 24], verbose=False)
        # Kiểm tra xem có bất kỳ độ trễ (lag) nào có p-value < 0.05 không
        has_causality = False
        for lag in gc_res:
            test_stat = gc_res[lag][0]['ssr_ftest']
            p_val = test_stat[1]
            if p_val < 0.05:
                has_causality = True
                break
        if has_causality:
            print(f" => {col} có quan hệ Granger Causality (p < 0.05).")
            selected_exog.append(col)
        else:
            print(f" => {col} KHÔNG có quan hệ Granger Causality.")
    except Exception as e:
         print(f" => Lỗi khi tính toán {col}: {e}")

print("\n=> Các biến ngoại sinh chính thức được chọn:", selected_exog)


--- Kết quả Granger Causality (Tìm p-value < 0.05) ---

Thử nghiệm: month_cos -> PM2.5
 => month_cos có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: surface_pressure -> PM2.5


 => surface_pressure có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: is_dry_season -> PM2.5
 => is_dry_season có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: wind_u -> PM2.5


 => wind_u có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: month -> PM2.5
 => month có quan hệ Granger Causality (p < 0.05).

=> Các biến ngoại sinh chính thức được chọn: ['month_cos', 'surface_pressure', 'is_dry_season', 'wind_u', 'month']


In [5]:
# Trích xuất dữ liệu Exogenous
train_exog_vals = train_X[selected_exog].values
val_exog_vals = val_X[selected_exog].values
test_exog_vals = test_X[selected_exog].values

train_y_vals = train_y.values
val_y_vals = val_y.values
test_y_vals = test_y.values


## 2. Kiểm định tính dừng - Augmented Dickey-Fuller (ADF)

Chúng ta sử dụng thuật toán kiểm định chuỗi thời gian để quyết định bậc d (differencing).

In [6]:
adf_result = adfuller(train_y_vals)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("p-value < 0.05 => Bác bỏ giả thuyết H0 (Chuỗi Không Dừng). Chuỗi dữ liệu LÀ DỪNG (Stationary). Ta có thể xét d=0.")
else:
    print("p-value >= 0.05 => Không thể bác bỏ H0. Chuỗi dữ liệu LÀ KHÔNG DỪNG (Non-Stationary). Cần thiết lập d=1 hoặc d=2.")


ADF Statistic: -6.471124386274397
p-value: 1.3663498557108741e-08
p-value < 0.05 => Bác bỏ giả thuyết H0 (Chuỗi Không Dừng). Chuỗi dữ liệu LÀ DỪNG (Stationary). Ta có thể xét d=0.


## 3. Auto-ARIMA & Residual Diagnostics

Với dữ liệu `m=24`, tìm kiếm toàn bộ lưới (Grid Search) sẽ tốn rất nhiều giờ. Thay vào đó `stepwise=True` sử dụng thuật toán tham lam tìm đường gần nhất tới điểm tối ưu dựa trên AIC.

Chúng ta sẽ chạy 4 cấu hình:
- **ARIMA:** Khởi nghiệm nhanh, không có Chu kỳ (seasonal=False), không Ngoại sinh (exogenous=None).
- **SARIMA:** Có chu kỳ `m=24`, không Ngoại sinh.
- **ARIMAX:** Không chu kỳ, Có Ngoại sinh.
- **SARIMAX:** Có chu kỳ, Có Ngoại sinh.

> **Tips:** Tùy vào cấu hình máy, mỗi cell Search dưới đây có thể tốn từ 2 -> 15 phút.

In [7]:
# Hàm helper hỗ trợ đo lường Metric
def regression_metrics(y_true, y_pred, inverse=True):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if inverse:
        y_true_inv = np.expm1(y_true)
        y_pred_inv = np.expm1(y_pred)
    else:
        y_true_inv = y_true
        y_pred_inv = y_pred
    rmse = float(np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    eps = 1e-6
    mask = np.abs(y_true_inv) > eps
    mape = np.mean(np.abs((y_true_inv[mask] - y_pred_inv[mask]) / y_true_inv[mask])) * 100.0
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

# ── True 24-Step Walk-Forward Forecast (No Data Leakage) ──────────────────────
def rolling_forecast_24h(fitted_model, all_y, start_idx, end_idx, all_exog=None):
    """
    Correct walk-forward evaluation for 24h-ahead forecasting.
    At index i, the target is PM2.5(T + 24). The most recent available true
    observation is PM2.5(T), which is safely located at index i - 24.
    We only feed observations up to i - 24 into the filter, and forecast 24 steps.
    """
    init_end = start_idx - 24
    
    # Initialize filter state up to the allowed historical boundary
    if all_exog is not None:
        current = fitted_model.apply(endog=all_y[:init_end], exog=all_exog[:init_end])
    else:
        current = fitted_model.apply(endog=all_y[:init_end])
        
    preds = []
    n = end_idx - start_idx
    for step, i in enumerate(range(start_idx, end_idx)):
        # The model's last observation is (i - 25). Extend by adding i - 24.
        if all_exog is not None:
            exog_step = all_exog[i-24 : i-23]
            current = current.append([all_y[i-24]], exog=exog_step, refit=False)
            # Persist exogenous attributes (e.g. weather at time T) across the 24-steps
            future_exog = np.tile(exog_step[0], (24, 1))
            fc = current.forecast(steps=24, exog=future_exog)
        else:
            current = current.append([all_y[i-24]], refit=False)
            fc = current.forecast(steps=24)
            
        val = float(fc.iloc[-1] if hasattr(fc, "iloc") else fc[-1])
        preds.append(val)
        
        if (step + 1) % 500 == 0:
            print(f"  rolling_forecast_24h: {step+1}/{n} done")
            
    return np.array(preds)

all_metrics = []
predictions_dict = {} # Key: (Model_Name, split), Value: Array

# Setup global sequences required by true 24-step out-of-sample mapping
all_y_global = np.concatenate([train_y_vals, val_y_vals, test_y_vals])
if 'selected_exog' in globals() and len(selected_exog) > 0:
    all_exog_global = np.vstack([train_exog_vals, val_exog_vals, test_exog_vals])
else:
    all_exog_global = None

val_start = len(train_y_vals)
val_end = val_start + len(val_y_vals)
test_start = val_end
test_end = test_start + len(test_y_vals)


In [8]:
# [1. ARIMA]
print("\n--- Tuning ARIMA (Non-Seasonal, No Exog) ---")
start = time.time()
model_arima = pm.auto_arima(train_y_vals, 
                            exogenous=None,
                            seasonal=False,
                            start_p=0, start_q=0, max_p=3, max_q=3,
                            d=None, trace=True, 
                            error_action='ignore', suppress_warnings=True, stepwise=True, random_state=RANDOM_STATE)
print(f"ARIMA tuned in {time.time()-start:.2f}s | Order: {model_arima.order} | AIC: {model_arima.aic():.2f}")

sm_arima = sm.tsa.ARIMA(endog=train_y_vals, order=model_arima.order).fit()

# In-sample train predictions remain equivalent to fit
predictions_dict[('ARIMA', 'train')] = sm_arima.fittedvalues

print("Rolling 24h forecast ARIMA on val set...")
predictions_dict[('ARIMA', 'val')]  = rolling_forecast_24h(sm_arima, all_y_global, val_start, val_end)
print("Rolling 24h forecast ARIMA on test set...")
predictions_dict[('ARIMA', 'test')] = rolling_forecast_24h(sm_arima, all_y_global, test_start, test_end)

lb_df = acorr_ljungbox(sm_arima.resid, lags=[24], return_df=True)
print("\nLjung-Box Test (ARIMA):")
display(lb_df)



--- Tuning ARIMA (Non-Seasonal, No Exog) ---
Performing stepwise search to minimize aic


 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-2469.853, Time=1.05 sec


 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=-2575.800, Time=0.95 sec


 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-2593.891, Time=1.06 sec


 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-2471.853, Time=0.35 sec


...

  rolling_forecast_24h: 1000/2128 done


  rolling_forecast_24h: 1500/2128 done


  rolling_forecast_24h: 2000/2128 done



Ljung-Box Test (ARIMA):


,lb_stat,lb_pvalue
24,194.325404,1.305172e-28


In [9]:
# [2. ARIMAX]
print("\n--- Tuning ARIMAX (Non-Seasonal, With Exog) ---")
start = time.time()
model_arimax = pm.auto_arima(train_y_vals, 
                             X=train_exog_vals,
                             seasonal=False,
                             start_p=0, start_q=0, max_p=3, max_q=3,
                             d=None, trace=True, 
                             error_action='ignore', suppress_warnings=True, stepwise=True, random_state=RANDOM_STATE)
print(f"ARIMAX tuned in {time.time()-start:.2f}s | Order: {model_arimax.order} | AIC: {model_arimax.aic():.2f}")

sm_arimax = sm.tsa.statespace.SARIMAX(endog=train_y_vals, exog=train_exog_vals, order=model_arimax.order, seasonal_order=(0,0,0,0)).fit(disp=False)

predictions_dict[('ARIMAX', 'train')] = sm_arimax.fittedvalues

print("Rolling 24h forecast ARIMAX on val set...")
predictions_dict[('ARIMAX', 'val')]  = rolling_forecast_24h(sm_arimax, all_y_global, val_start, val_end, all_exog_global)
print("Rolling 24h forecast ARIMAX on test set...")
predictions_dict[('ARIMAX', 'test')] = rolling_forecast_24h(sm_arimax, all_y_global, test_start, test_end, all_exog_global)



--- Tuning ARIMAX (Non-Seasonal, With Exog) ---
Performing stepwise search to minimize aic


 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-2576.389, Time=2.02 sec


 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=-2655.610, Time=3.78 sec


 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-2672.226, Time=5.06 sec


 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-2578.389, Time=2.42 sec


...

Rolling 24h forecast ARIMAX on test set...


  rolling_forecast_24h: 500/2128 done


  rolling_forecast_24h: 1000/2128 done


  rolling_forecast_24h: 1500/2128 done


  rolling_forecast_24h: 2000/2128 done


In [10]:
# [3. SARIMA]
print("\n--- Tuning SARIMA (Seasonal m=24, No Exog) ---")
start = time.time()
model_sarima = pm.auto_arima(train_y_vals, 
                             exogenous=None,
                             seasonal=True, m=24,
                             start_p=0, start_q=0, max_p=2, max_q=2,
                             start_P=0, start_Q=0, max_P=1, max_Q=1,
                             d=None, D=None, trace=True, 
                             error_action='ignore', suppress_warnings=True, stepwise=True, random_state=RANDOM_STATE)
print(f"SARIMA tuned in {time.time()-start:.2f}s | Order: {model_sarima.order} | Seasonal: {model_sarima.seasonal_order} | AIC: {model_sarima.aic():.2f}")

sm_sarima = sm.tsa.statespace.SARIMAX(endog=train_y_vals, order=model_sarima.order, seasonal_order=model_sarima.seasonal_order).fit(disp=False)

predictions_dict[('SARIMA', 'train')] = sm_sarima.fittedvalues

print("Rolling 24h forecast SARIMA on val set...")
predictions_dict[('SARIMA', 'val')]  = rolling_forecast_24h(sm_sarima, all_y_global, val_start, val_end)
print("Rolling 24h forecast SARIMA on test set...")
predictions_dict[('SARIMA', 'test')] = rolling_forecast_24h(sm_sarima, all_y_global, test_start, test_end)



--- Tuning SARIMA (Seasonal m=24, No Exog) ---
Performing stepwise search to minimize aic


 ARIMA(0,1,0)(0,0,0)[24] intercept   : AIC=-2469.853, Time=0.45 sec


 ARIMA(1,1,0)(1,0,0)[24] intercept   : AIC=-2660.595, Time=3.43 sec


 ARIMA(0,1,1)(0,0,1)[24] intercept   : AIC=-2663.327, Time=5.82 sec
 ARIMA(0,1,0)(0,0,0)[24]             : AIC=-2471.853, Time=0.21 sec


 ARIMA(0,1,1)(0,0,0)[24] intercept   : AIC=-2593.891, Time=0.49 sec


...

Rolling 24h forecast SARIMA on test set...


  rolling_forecast_24h: 500/2128 done


  rolling_forecast_24h: 1000/2128 done


  rolling_forecast_24h: 1500/2128 done


  rolling_forecast_24h: 2000/2128 done


In [11]:
# [4. SARIMAX]
print("\n--- Tuning SARIMAX (Seasonal m=24, With Exog) ---")
start = time.time()
model_sarimax = pm.auto_arima(train_y_vals, 
                              X=train_exog_vals,
                              seasonal=True, m=24,
                              start_p=0, start_q=0, max_p=2, max_q=2,
                              start_P=0, start_Q=0, max_P=1, max_Q=1,
                              d=None, D=None, trace=True, 
                              error_action='ignore', suppress_warnings=True, stepwise=True, random_state=RANDOM_STATE)
print(f"SARIMAX tuned in {time.time()-start:.2f}s | Order: {model_sarimax.order} | Seasonal: {model_sarimax.seasonal_order} | AIC: {model_sarimax.aic():.2f}")

sm_sarimax = sm.tsa.statespace.SARIMAX(endog=train_y_vals, exog=train_exog_vals, order=model_sarimax.order, seasonal_order=model_sarimax.seasonal_order).fit(disp=False)

predictions_dict[('SARIMAX', 'train')] = sm_sarimax.fittedvalues

print("Rolling 24h forecast SARIMAX on val set...")
predictions_dict[('SARIMAX', 'val')]  = rolling_forecast_24h(sm_sarimax, all_y_global, val_start, val_end, all_exog_global)
print("Rolling 24h forecast SARIMAX on test set...")
predictions_dict[('SARIMAX', 'test')] = rolling_forecast_24h(sm_sarimax, all_y_global, test_start, test_end, all_exog_global)

lb_df_sarimax = acorr_ljungbox(sm_sarimax.resid, lags=[24], return_df=True)
print("\nLjung-Box Test (SARIMAX):")
display(lb_df_sarimax)



--- Tuning SARIMAX (Seasonal m=24, With Exog) ---
Performing stepwise search to minimize aic


 ARIMA(0,1,0)(0,0,0)[24] intercept   : AIC=-2576.389, Time=1.46 sec


 ARIMA(1,1,0)(1,0,0)[24] intercept   : AIC=-2719.686, Time=17.95 sec


 ARIMA(0,1,1)(0,0,1)[24] intercept   : AIC=-2725.589, Time=22.00 sec


 ARIMA(0,1,0)(0,0,0)[24]             : AIC=-2578.389, Time=1.38 sec


...

  rolling_forecast_24h: 1000/2128 done


  rolling_forecast_24h: 1500/2128 done


  rolling_forecast_24h: 2000/2128 done



Ljung-Box Test (SARIMAX):


,lb_stat,lb_pvalue
24,83.019187,1.995694e-08


## 4. Leaderboard Nhóm Thống Kê & Lưu Xuất

So sánh thông số thực tế của cấu hình tối ưu thu được và lưu file dưới dạng numpy Array `.npy`

In [12]:
# Tính metrics — regression_metrics() already applies expm1 (inverse=True),
# so RMSE/MAE/MAPE below are in REAL µg/m³ units, comparable to tree/DL models.
for name in ['ARIMA', 'ARIMAX', 'SARIMA', 'SARIMAX']:
    all_metrics.append({'model': name, 'split': 'train', **regression_metrics(train_y_vals, predictions_dict[(name, 'train')])})
    all_metrics.append({'model': name, 'split': 'val',   **regression_metrics(val_y_vals,   predictions_dict[(name, 'val')])})
    all_metrics.append({'model': name, 'split': 'test',  **regression_metrics(test_y_vals,  predictions_dict[(name, 'test')])})

metric_df = pd.DataFrame(all_metrics)

print("--- LEADERBOARD BẢN TUNED TRÊN TEST-SET (units: µg/m³ after expm1) ---")
display(metric_df[metric_df['split'] == 'test'].set_index('model')[['RMSE', 'MAE', 'MAPE']].round(4))

# ── Sanity check: confirm pred[t] ≠ y_true[t-1] ──────────────────────────────
test_preds_best = predictions_dict[('SARIMAX', 'test')]
shift_corr = float(np.corrcoef(test_preds_best[1:], test_y_vals[:-1])[0, 1])
print(f"\nSanity check — corr(pred[t], y_true[t-1]) = {shift_corr:.4f}")
print("  → Should be close to 0 (not ~0.87) if no shift-leakage.")

# Cài đặt mô hình siêu việt nhất thuộc nhóm Stats
best_stat_model = "SARIMAX" # Thông thường thuộc SARIMAX/SARIMA.


--- LEADERBOARD BẢN TUNED TRÊN TEST-SET (units: µg/m³ after expm1) ---


,RMSE,MAE,MAPE
model,,,
ARIMA,18.7909,13.2711,34.8067
ARIMAX,18.7623,13.3715,34.9846
SARIMA,18.4077,12.8573,33.5456
SARIMAX,17.3994,11.9630,30.9846



Sanity check — corr(pred[t], y_true[t-1]) = 0.4648
  → Should be close to 0 (not ~0.87) if no shift-leakage.


In [13]:
# Export To Pickle
# NOTE: predictions stored in LOG-SPACE (log1p / target_pm25_h24_log).
#       To get real µg/m³: np.expm1(pred_array)
#       Metrics in the leaderboard above are already inverted (real units).
preds_export = {
    "arima": {
        "train": predictions_dict[("ARIMA", "train")],
        "val":   predictions_dict[("ARIMA", "val")],
        "test":  predictions_dict[("ARIMA", "test")],
    },
    "arimax": {
        "train": predictions_dict[("ARIMAX", "train")],
        "val":   predictions_dict[("ARIMAX", "val")],
        "test":  predictions_dict[("ARIMAX", "test")],
    },
    "sarima": {
        "train": predictions_dict[("SARIMA", "train")],
        "val":   predictions_dict[("SARIMA", "val")],
        "test":  predictions_dict[("SARIMA", "test")],
    },
    "sarimax": {
        "train": predictions_dict[("SARIMAX", "train")],
        "val":   predictions_dict[("SARIMAX", "val")],
        "test":  predictions_dict[("SARIMAX", "test")],
    }
}

# Also export real-unit (µg/m³) predictions for direct comparison with tree/DL
preds_export_real = {
    model_name: {
        split: np.expm1(arr)
        for split, arr in splits.items()
    }
    for model_name, splits in preds_export.items()
}

pickle_path = PRED_DIR / "tuned_stats_preds.pkl"
with open(pickle_path, 'wb') as handle:
    pickle.dump(preds_export, handle, protocol=pickle.HIGHEST_PROTOCOL)

pickle_path_real = PRED_DIR / "tuned_stats_preds_real_units.pkl"
with open(pickle_path_real, 'wb') as handle:
    pickle.dump(preds_export_real, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved log-space predictions  → {pickle_path}")
print(f"Saved real-unit predictions  → {pickle_path_real}")


Saved log-space predictions  → ..\outputs\predictions\tuned_stats_preds.pkl
Saved real-unit predictions  → ..\outputs\predictions\tuned_stats_preds_real_units.pkl
